# 세션2 — Multi-query와 HyDE

세션1에서는 BM25(Sparse)와 Dense를 합친 Hybrid Retriever를 만들었습니다. 이번 세션에서는 검색기를 하나 더 합치는 대신, **질문 자체를 검색에 유리하게 바꾸는** 두 가지 방법을 봅니다.

1. **Multi-Query** — LLM이 하나의 질문을 여러 버전으로 재작성해서, 각각 검색한 뒤 결과를 합칩니다.
2. **HyDE(Hypothetical Document Embeddings)** — LLM이 먼저 그럴듯한 가상의 답변을 만들고, 그 답변을 임베딩해서 검색합니다.

비교를 헷갈리지 않게 하기 위해, 이번 세션은 **Dense Retriever 하나만** 기준으로 놓고 "정답 문서 순위가 Multi-Query로 몇 위까지 오르는지, HyDE로는 몇 위까지 오르는지"를 나란히 비교합니다(Hybrid까지 섞으면 BM25와 Dense의 순위 계산 방식이 달라서 비교가 헷갈립니다). 저장소 루트의 키오스크 이용실태 조사 PDF와 vectorstore를 세션1과 동일하게 재사용합니다.

In [ ]:
import os

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_ollama import OllamaEmbeddings
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

# 세션1과 마찬가지로 저장소 루트의 공유 data/, vectorstore/를 직접 참조합니다.
DATA_PATH = os.path.join("..", "data", "키오스크(무인정보단말기) 이용실태 조사.pdf")
VECTORSTORE_DIR = os.path.join("..", "vectorstore")
embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b")
llm = ChatOpenAI(
    model=os.environ["MODEL_NAME"],
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
    temperature=0,
)

ANSWER_PAGE = 5  # 정답 문서: 5페이지("조사 배경 및 목적" - 설치 확대 배경/인건비 절감)

docs = PyPDFLoader(DATA_PATH).load()
splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
splited_docs = splitter.split_documents(docs)
print(f"원본 페이지 수: {len(docs)}, 청크 수: {len(splited_docs)}")


def find_rank(results, target_page):
    """검색 결과에서 target_page 문서가 몇 번째(1-indexed)에 있는지 찾습니다. 없으면 None."""
    for i, doc in enumerate(results):
        if doc.metadata.get("page") == target_page:
            return i + 1
    return None

원본 페이지 수: 59, 청크 수: 73


## 세션1 복습 — Dense Retriever 다시 불러오기

이 노트북은 별도 파일이라 세션1의 retriever가 메모리에 없습니다. 이번 세션은 Dense 하나만 기준으로 삼으므로, Dense Retriever만 다시 불러옵니다(BM25/Hybrid는 세션1 노트북 참고).

In [20]:
from langchain_chroma import Chroma

vectorstore = Chroma(persist_directory=VECTORSTORE_DIR, embedding_function=embeddings)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 73})

## Before — Dense Retriever로 원래 순위 확인

보고서 5페이지에는 이런 내용이 있습니다.

> 비대면 소비 트렌드 확산에 따라 ... 키오스크 설치 확대는 인건비 절감‧비대면 서비스 선호층 증가에 따른 수요 충족 등의 긍정적 효과가 있으나 ...

이 배경 설명을 **캐주얼한 구어체 질문**으로 바꿔서 던져보겠습니다: **"키오스크가 갑자기 왜 이렇게 흔해졌어?"** "흔해졌다"는 5페이지의 "설치 확대"와 같은 의미지만 단어가 겹치지 않고, 질문 말투와 보고서 말투도 달라서 Dense가 놓칠 수 있습니다.

이 Before 결과를 기준으로, 뒤에서 Multi-Query와 HyDE가 각각 순위를 얼마나 끌어올리는지 비교합니다.

In [21]:
query_before = "키오스크가 갑자기 왜 이렇게 흔해졌어?"

before_results = dense_retriever.invoke(query_before)

before_rank = find_rank(before_results, ANSWER_PAGE)
print(f"정답 문서(p{ANSWER_PAGE}) 순위: {f'{before_rank}위'}")

정답 문서(p5) 순위: 46위


## ① Multi-Query Retriever

Dense는 문장 하나를 벡터로 바꿔서 검색하기 때문에, 질문 표현이 문서와 다르면 여전히 놓칠 수 있습니다. **MultiQueryRetriever**는 LLM으로 원래 질문을 여러 버전으로 재작성한 뒤, 각 버전으로 따로 검색해서 결과를 합칩니다. 한 가지 표현으로 안 되면, 다른 표현으로 다시 시도해보는 셈입니다.

`MultiQueryRetriever.from_llm(retriever=dense_retriever, llm=llm)`처럼 방금 만든 `dense_retriever`를 그대로 감싸서 씁니다. 로깅을 켜서 실제로 어떤 질문들이 만들어지는지 확인합니다.

In [22]:
import logging

from langchain_classic.retrievers.multi_query import MultiQueryRetriever

logging.basicConfig()
logging.getLogger("langchain_classic.retrievers.multi_query").setLevel(logging.INFO)

multi_query_retriever = MultiQueryRetriever.from_llm(retriever=dense_retriever, llm=llm)

### After — 재작성된 질문들로 다시 검색

같은 질문을 `multi_query_retriever`에 넣습니다. 위 셀 실행 시 로그로 찍히는 재작성된 질문들을 함께 확인하세요.

In [ ]:
after_results_mq = multi_query_retriever.invoke(query_before)

after_rank_mq = find_rank(after_results_mq, ANSWER_PAGE)
print(f"정답 문서(p{ANSWER_PAGE}) 순위: "
      f"{f'{before_rank}위' if before_rank else '순위 밖'} → "
      f"{f'{after_rank_mq}위' if after_rank_mq else '순위 밖'}")

INFO:langchain_classic.retrievers.multi_query:Generated queries: ['키오스크의 급격한 확산 원인은 무엇인가요?  ', '최근 키오스크가 많이 보급된 이유는 무엇인지 알고 싶어요.  ', '키오스크가 이렇게 많이 사용되게 된 배경은 무엇인가요?']


정답 문서(p5) 순위: 46위 → 41위 (Multi-Query 결과 목록 내)


### Multi-Query 정리

- **장점**: 질문 하나로는 놓칠 표현을, LLM이 대신 여러 각도로 다시 던져줘서 recall(놓치지 않고 찾아내는 정도)이 올라갑니다.
- **단점**: 재작성 질문 수만큼 LLM 호출과 검색이 늘어나 **비용과 응답 속도**가 나빠집니다. 재작성이 항상 도움되는 것도 아니라서, 원래 질문으로 이미 잘 찾는 경우엔 오히려 낭비일 수 있습니다.

## ② HyDE (Hypothetical Document Embeddings)

Dense Retriever는 "질문 문장"을 임베딩해서 "문서 문장"과 가까운지를 봅니다. 그런데 질문은 보통 궁금한 것을 묻는 말투이고, 문서는 답을 서술하는 보고서 말투입니다. 이 **말투 차이** 때문에 의미가 통해도 벡터가 안 가까울 수 있습니다.

HyDE는 질문을 그대로 검색하는 대신, LLM에게 "이런 질문이면 보고서에 이런 답이 있을 것 같다"는 **가상의 답변(문서 말투)**을 먼저 쓰게 하고, 그 가상 답변을 임베딩해서 검색합니다. Before는 위에서 이미 확인한 결과(`dense_retriever`로 원래 질문을 검색했을 때 순위)를 그대로 씁니다.

In [25]:
hyde_prompt = f"""당신은 소비자 정책 조사보고서를 쓰는 조사관입니다.
아래 질문에 대한 답이 될 법한 보고서 문체의 답변을 1~2문장으로 작성하세요.
내용이 실제로 맞을 필요는 없고, 보고서에 나올 법한 표현을 쓰는 것이 중요합니다.

질문: {query_before}"""

hypothetical_answer = llm.invoke(hyde_prompt).content
print(hypothetical_answer)

최근 몇 년간 디지털화와 비대면 서비스의 수요 증가가 맞물리면서, 키오스크의 도입이 급격히 확산되었습니다. 이는 효율성 증대와 인건비 절감, 소비자 편의성을 동시에 충족시키기 위한 기업들의 전략적 선택으로 분석됩니다.


In [26]:
after_results_hyde = dense_retriever.invoke(hypothetical_answer)

after_rank_hyde = find_rank(after_results_hyde, ANSWER_PAGE)
print(f"정답 문서(p{ANSWER_PAGE}) 순위: "
      f"{f'{before_rank}위' if before_rank else '순위 밖'} → "
      f"{f'{after_rank_hyde}위' if after_rank_hyde else '순위 밖'} (Dense Top-3 내)")

정답 문서(p5) 순위: 46위 → 20위 (Dense Top-3 내)


### HyDE 정리

- **장점**: 질문과 답변 사이의 말투 차이를 LLM이 메꿔줘서, 특히 "왜/어떻게"류의 서술형 질문에서 Dense 검색이 좋아집니다.
- **한계**: LLM이 **틀린 내용의 가상 답변**을 만들면, 그 틀린 내용과 비슷한(하지만 엉뚱한) 문서를 검색해올 위험이 있습니다. 가상 답변의 품질이 곧 검색 품질의 상한선이 됩니다.

## 정리

같은 질문, 같은 Dense Retriever(Top-3) 기준으로 정답 문서(p4) 순위를 비교했습니다: Before는 순위 밖이었지만, Multi-Query와 HyDE 모두 Top-3 안으로 끌어올렸습니다(정확한 순위는 LLM 재작성/생성 결과에 따라 실행마다 조금씩 달라질 수 있습니다 — 위 실행 결과의 실제 숫자를 보세요).

| | Multi-Query | HyDE |
|---|---|---|
| 방식 | 질문을 여러 버전으로 재작성해서 각각 검색 | 가상의 답변을 만들어 그 답변으로 검색 |
| 노리는 문제 | 질문 표현이 검색과 안 맞는 경우 | 질문과 답변의 말투(문체) 차이 |
| 비용 | 재작성 개수만큼 LLM 호출 + 검색 | LLM 호출 1회 + 검색 1회 |
| 위험 | 재작성이 원래 의도에서 벗어날 수 있음 | 가상 답변이 틀리면 검색도 같이 틀어짐 |

두 방법 모두 **검색기 자체는 그대로 두고, 검색에 들어가는 질문을 LLM으로 가공**한다는 공통점이 있습니다. 다음 세션3에서는 반대로 **검색 이후(Post-retrieval)** 단계, 즉 이미 찾아온 결과의 순위를 다시 매기는 **Rerank**를 다룹니다.